> **Public copy.** This step reads the raw Jira export and the test trackers, which stay private, so it cannot be run from this repository. The outputs below are from the original run. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E, releases such as 1.51, System tag 1 to 11 and so on), so every count is exactly as computed.


# 01 Load and check the Jira export

**Goal:** load the Jira export, check it is complete, and build a `defects` table with clear flags for what counts in the analysis.

**Input:** `data/private/hindsight_work.sqlite` (from `scripts/jira_export.py`, run on 20 Nov 2026).
**Output:** table `defects` in `data/private/hindsight_clean.sqlite`.

Rule: we never delete rows. We add flags, so every choice can be seen and changed later.

In [1]:
import sqlite3, sys
from pathlib import Path
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
WORK = ROOT / 'data' / 'private' / 'hindsight_work.sqlite'    # Jira export (read only)
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', WORK)

Reading from: data/private/hindsight_work.sqlite


## 1. What is in the export?

In [2]:
con = sqlite3.connect(WORK)
tables = pd.read_sql("select name from sqlite_master where type='table'", con).name
pd.DataFrame({'table': tables, 'rows': [con.execute(f'select count(*) from "{t}"').fetchone()[0] for t in tables]})

,table,rows
0,issues,7648
1,labels,16592
2,fix_versions,3782
3,affects_versions,81
4,comments,26038
5,changelog,110405
6,issue_links,1665
7,project_versions,87
8,defect_reopens,3152


In [3]:
issues = pd.read_sql('select * from issues', con)
issues.issue_type.value_counts()

issue_type
Task      4251
Defect    3152
Story      154
Epic        90
Risk         1
Name: count, dtype: int64

Expected on 20 Nov 2026: 7,648 issues, of which 3,152 are Defects. We keep only Defects from here on.

In [4]:
d = issues[issues.issue_type == 'Defect'].copy()
assert len(d) == 3152, len(d)
for col in ['created', 'updated', 'resolution_date']:
    d[col] = pd.to_datetime(d[col], utc=True, errors='coerce')
d['created_year'] = d.created.dt.year
d.created_year.value_counts().sort_index()

created_year
2016    150
2017    201
2018    285
2019    415
2020    328
2021    285
2022    233
2023    248
2024    448
2025    330
2026    229
Name: count, dtype: int64

## 2. Flags: what counts as a real defect?

- **withdrawn**: status Withdrawn (WTR) or Deferred, or resolution Withdrawn. Planned in the proposal: these are left out of risk counts.
- **not_a_defect**: the root cause says it was not a product fault (As per Design, Not an issue, Not reproduced), or the ticket is a request or a dummy (deployment request, repoint request, dummy defect).
- **perf_defect**: performance and NFT defects. Performance testing is out of scope (agreed 23 Nov 2026), so these are counted but not used for risk.
- **open**: not yet in a Done-category status.
- **counts_for_risk**: a real, valid, in-scope defect, used in the risk counts.

In [5]:
d.status.value_counts()

status
Done                  2568
To Be Started          214
Withdrawn (WTR)        177
Triage                  56
Backlog                 46
QA Sign Off             30
In Development          28
Monitoring               9
Deferred                 8
Ready for Test           7
Blocked (migrated)       5
In QA                    4
Name: count, dtype: int64

In [6]:
d.cf_defect_root_cause.fillna('(empty)').value_counts()

cf_defect_root_cause
(empty)                             1163
Code                                 563
Environment                          482
Configuration                        295
As per Design                        104
Not an issue                          95
Build                                 72
Documentation                         58
Not reproduced                        50
Not known/Other                       42
Data                                  41
Infrastructure                        39
Deployment                            31
Design                                30
Requirement                           30
Others                                20
Existing Production                   13
Release                                7
Duplicate                              5
Converted to Change Request            4
Tester Error / User Error              3
Test Case invalid                      2
Communication / Miscommunication       2
Analysis                            

In [7]:
NOT_DEFECT = {'As per Design', 'Not an issue', 'Not reproduced'}
text = (d.summary.fillna('') + ' ' + d.labels.fillna('')).str.lower()
d['withdrawn'] = d.status.isin(['Withdrawn (WTR)', 'Deferred']) | (d.resolution == 'Withdrawn')
d['request_not_defect'] = text.str.contains(r'deployment ?request|dummy defect|repoint ?request')
d['not_a_defect'] = d.cf_defect_root_cause.isin(NOT_DEFECT) | d.request_not_defect
d['perf_defect'] = text.str.contains(r'\bperf|\bnft\b|jmeter')
d['open'] = d.status_category != 'Done'
d['accepted_as_is'] = d.resolution == 'Accepted'
d['counts_for_risk'] = ~d.withdrawn & ~d.not_a_defect & ~d.perf_defect
d[['withdrawn', 'not_a_defect', 'request_not_defect', 'perf_defect', 'open', 'accepted_as_is', 'counts_for_risk']].sum()

withdrawn              192
not_a_defect           261
request_not_defect      12
perf_defect            169
open                   369
accepted_as_is         136
counts_for_risk       2597
dtype: int64

## 3. Reopens (from the change history, built during the export)

In [8]:
reopens = pd.read_sql('select * from defect_reopens', con)
d = d.merge(reopens[['issue_key', 'reopen_events_strict', 'reopen_events_wide']],
            left_on='key', right_on='issue_key', how='left').drop(columns='issue_key')
(d.reopen_events_strict > 0).sum(), (d.reopen_events_wide > 0).sum()

(np.int64(75), np.int64(114))

## 4. Save the `defects` table

In [9]:
KEEP = ['key', 'summary', 'status', 'status_category', 'resolution', 'priority', 'created', 'resolution_date',
        'created_year', 'project_name', 'security_level', 'environment', 'external_ticket_ref', 'labels',
        'cf_defect_root_cause', 'withdrawn', 'not_a_defect', 'perf_defect', 'open', 'accepted_as_is', 'counts_for_risk',
        'reopen_events_strict', 'reopen_events_wide', 'reporter', 'assignee']
out = d[KEEP].rename(columns={'cf_defect_root_cause': 'root_cause'})
for col in ['created', 'resolution_date']:
    out[col] = out[col].dt.strftime('%Y-%m-%d %H:%M:%S')
with sqlite3.connect(CLEAN) as cc:
    out.to_sql('defects', cc, if_exists='replace', index=False)
print(len(out), 'defects saved;', int(out.counts_for_risk.sum()), 'count for risk')

3152 defects saved; 2597 count for risk
